# Group Activity Recognition - Baseline 3
This notebook contains the complete, self-contained implementation of Baseline 3 (Temporal Model with Image Features) for Group Activity Recognition on the Volleyball dataset.

### Pipeline Components:
1. **Configurations**: Class names and indices mapping.
2. **Dataset**: Unified dataset class (`VolleyBallDataset`) configured in sequence/clip mode loading 9-frame sequences per clip.
3. **Model**: ResNet-50 backbone extracting features per frame, processed by a single-layer LSTM over the sequence, and classified using a group classification head.
4. **Utilities**: Evaluation metrics (Macro F1-score).
5. **Training/Validation**: Single epoch training/eval loops.
6. **Execution**: Training initialization and main runner function.

In [17]:
import os
import sys
import cv2
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.notebook import tqdm

## 1. Configurations
Define the categories and lookup indices for both the group activities (8 classes) and individual player actions (9 classes).

In [18]:
# Group Activity Categories (8 classes)
GROUP_ACTIVITIES = ['r_set', 'r_spike', 'r_pass', 'r_winpoint', 'l_set', 'l_spike', 'l_pass', 'l_winpoint']
# Individual Player Action Categories (9 classes)
PLAYER_ACTIONS = ['waiting', 'setting', 'spiking', 'digging', 'jumping', 'blocking', 'falling', 'moving', 'standing']

group_to_idx = {name: idx for idx, name in enumerate(GROUP_ACTIVITIES)}
action_to_idx = {name: idx for idx, name in enumerate(PLAYER_ACTIONS)}

## 2. Dataset Loader
The unified `VolleyBallDataset` class loads frame sequences centered on target frames if `seq_len` is provided; otherwise, it loads single frames. If the dataset path is not found, it raises a FileNotFoundError.

In [ ]:
class VolleyBallDataset(Dataset):
    def __init__(self, split='train', transform=None, data_path='./volleyball/volleyball_/videos', seq_len=None, stride=3):
        self.data_path = data_path
        self.split = split
        self.transform = transform
        self.seq_len = seq_len
        self.stride = stride
        
        # Define video splits
        self.train_videos = {1, 3, 6, 7, 10, 13, 15, 16, 18, 22, 23, 31, 32, 36, 38, 39, 40, 41, 42, 48, 50, 52, 53, 54}
        self.val_videos = {0, 2, 8, 12, 17, 19, 24, 26, 27, 28, 30, 33, 46, 49, 51}
        self.test_videos = {4, 5, 9, 11, 14, 20, 21, 25, 29, 34, 35, 37, 43, 44, 45, 47}
        
        self.data = []
        self.load_data()

    def load_data(self):
        if not os.path.exists(self.data_path):
            raise FileNotFoundError(f"Data path '{self.data_path}' not found.")
            
        for video_folder in os.listdir(self.data_path):
            if not video_folder.isdigit():
                continue
            video_id = int(video_folder)
            
            # Filter by split
            if self.split == 'train' and video_id not in self.train_videos:
                continue
            elif self.split == 'val' and video_id not in self.val_videos:
                continue
            elif self.split == 'test' and video_id not in self.test_videos:
                continue
                
            video_path = os.path.join(self.data_path, video_folder)
            if not os.path.isdir(video_path):
                continue
            annotation_file = os.path.join(video_path, 'annotations.txt')
            if not os.path.exists(annotation_file):
                continue
            
            for row in open(annotation_file, 'r'):
                frame_folder, groupLabel, *playersAnnotations = row.strip().split(' ')
                frame_name = frame_folder.split('.')[0]
                image_frame  = os.path.join(video_path, frame_name, frame_name + '.jpg')
                
                parsed_players = [{
                    'x': int(playersAnnotations[i]),
                    'y': int(playersAnnotations[i+1]),
                    'w': int(playersAnnotations[i+2]),
                    'h': int(playersAnnotations[i+3]),
                    'action': playersAnnotations[i+4]
                } for i in range(0, len(playersAnnotations), 5)]
                
                annotation = {
                    'groupLabel': groupLabel,
                    'playersAnnotations': parsed_players
                }
                self.data.append((image_frame, annotation))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_frame, annotation = self.data[idx]
        
        # Sequence mode
        if self.seq_len is not None and self.seq_len > 1:
            # We are reading from disk
            frame_dir = os.path.dirname(image_frame)
            frame_name = os.path.basename(image_frame).split('.')[0]
            
            # List all frames inside the directory and sort them chronologically
            all_frames = sorted([f for f in os.listdir(frame_dir) if f.endswith('.jpg') or f.endswith('.png')])
            
            # Find the annotated frame index in the list of frames
            target_file = os.path.basename(image_frame)
            if target_file in all_frames:
                mid_idx = all_frames.index(target_file)
            else:
                mid_idx = len(all_frames) // 2
                
            # Sample seq_len frames centered around mid_idx
            half_len = self.seq_len // 2
            offsets = [i * self.stride for i in range(-half_len, half_len + 1)]
            
            images = []
            for offset in offsets:
                sample_idx = mid_idx + offset
                # Clamp to index boundaries [0, len(all_frames) - 1]
                sample_idx = max(0, min(sample_idx, len(all_frames) - 1))
                frame_path = os.path.join(frame_dir, all_frames[sample_idx])
                
                image = cv2.imread(frame_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                images.append(image)

            # Scale annotations based on the first frame's original size
            h_orig, w_orig = images[0].shape[:2]
            target_h, target_w = 720, 1280
            
            scale_x = target_w / w_orig
            scale_y = target_h / h_orig
            
            scaled_players = []
            for player in annotation['playersAnnotations']:
                x = int(player['x'] * scale_x)
                y = int(player['y'] * scale_y)
                w = int(player['w'] * scale_x)
                h = int(player['h'] * scale_y)
                
                action_str = player['action']
                action_idx = action_to_idx.get(action_str.lower(), 0) if not str(action_str).isdigit() else int(action_str)
                
                scaled_players.append({
                    'x': x,
                    'y': y,
                    'w': w,
                    'h': h,
                    'action': action_str,
                    'action_idx': action_idx
                })
                
            group_str = annotation['groupLabel']
            group_idx = group_to_idx.get(group_str.lower(), 0) if not str(group_str).isdigit() else int(group_str)
            
            scaled_annotation = {
                'groupLabel': group_str,
                'groupLabel_idx': group_idx,
                'playersAnnotations': scaled_players
            }
            
            # Apply transformation to each frame
            processed_images = []
            for img in images:
                if h_orig != target_h or w_orig != target_w:
                    img = cv2.resize(img, (target_w, target_h))
                    
                if self.transform:
                    from PIL import Image
                    img = Image.fromarray(img)
                    img = self.transform(img)
                else:
                    img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
                processed_images.append(img)
                
            # Stack images along sequence dimension: (seq_len, C, H, W)
            images_tensor = torch.stack(processed_images, dim=0)
            
            return images_tensor, scaled_annotation

        # Single frame mode (seq_len is None or 1)
        else:
            image = cv2.imread(image_frame)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                
            h_orig, w_orig = image.shape[:2]
            target_h, target_w = 720, 1280
            
            scale_x = target_w / w_orig
            scale_y = target_h / h_orig
            
            scaled_players = []
            for player in annotation['playersAnnotations']:
                x = int(player['x'] * scale_x)
                y = int(player['y'] * scale_y)
                w = int(player['w'] * scale_x)
                h = int(player['h'] * scale_y)
                
                action_str = player['action']
                action_idx = action_to_idx.get(action_str.lower(), 0) if not str(action_str).isdigit() else int(action_str)
                
                scaled_players.append({
                    'x': x,
                    'y': y,
                    'w': w,
                    'h': h,
                    'action': action_str,
                    'action_idx': action_idx
                })
                
            group_str = annotation['groupLabel']
            group_idx = group_to_idx.get(group_str.lower(), 0) if not str(group_str).isdigit() else int(group_str)
            
            scaled_annotation = {
                'groupLabel': group_str,
                'groupLabel_idx': group_idx,
                'playersAnnotations': scaled_players
            }
            
            if h_orig != target_h or w_orig != target_w:
                image = cv2.resize(image, (target_w, target_h))
                
            if self.transform:
                from PIL import Image
                image = Image.fromarray(image)
                image = self.transform(image)
                
            return image, scaled_annotation

## 3. Model Architecture
The baseline 3 model is a temporal model processing 9-frame sequence clips. Features are extracted per frame using ResNet-50, fed into a single-layer LSTM, and classified using group activity fully connected layers.

In [20]:
class GroupActivityRecognitionB3(nn.Module):
    def __init__(self, num_group_classes=8, num_action_classes=9, embed_dim=2048, hidden_size=512, num_layers=1, dropout=0.3):
        super(GroupActivityRecognitionB3, self).__init__()
        self.num_group_classes = num_group_classes
        self.num_action_classes = num_action_classes
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        # Load ResNet-50 backbone
        try:
            self.resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        except Exception:
            self.resnet = models.resnet50(pretrained=True)
            
        # Replace the final fully connected layer of ResNet-50 with Identity to extract 2048-dim features
        self.resnet.fc = nn.Identity()
        
        # LSTM layer to model temporal sequence of image features
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False
        )
        
        # Classifier for group activities fed by the final timestep output of the LSTM
        if dropout > 0:
            self.classifier = nn.Sequential(
                nn.Dropout(p=dropout),
                nn.Linear(hidden_size, num_group_classes)
            )
        else:
            self.classifier = nn.Linear(hidden_size, num_group_classes)

    def forward(self, images, annotations=None):
        batch_size, seq_len, C, H, W = images.shape
        device = images.device
        
        # Reshape to (B * seq_len, C, H, W) for batch computation through backbone
        images_flat = images.view(batch_size * seq_len, C, H, W)
        
        # Extract features for all frames in the batch
        features_flat = self.resnet(images_flat)  # (B * seq_len, 2048)
        
        # Reshape back to sequence format: (B, seq_len, 2048)
        features = features_flat.view(batch_size, seq_len, -1)
        
        # Pass sequence features through LSTM
        lstm_out, (hn, cn) = self.lstm(features)  # lstm_out: (B, seq_len, hidden_size)
        
        # Select output from the final timestep
        out = lstm_out[:, -1, :]  # (B, hidden_size)
        
        # Classify group activity
        group_output = self.classifier(out)
        
        # Return empty action predictions for compatibility with training/validation loops
        action_output = torch.zeros(0, self.num_action_classes, device=device)
        return group_output, action_output

## 4. Evaluation Utilities
Helper function to compute Macro F1-score for model evaluation.

In [21]:
try:
    from sklearn.metrics import f1_score
    def compute_macro_f1(preds, targets):
        return f1_score(targets, preds, average='macro', zero_division=0)
except ImportError:
    def compute_macro_f1(preds, targets):
        classes = np.unique(np.concatenate([preds, targets]))
        if len(classes) == 0:
            return 0.0
        f1s = []
        for c in classes:
            tp = np.sum((preds == c) & (targets == c))
            fp = np.sum((preds == c) & (targets != c))
            fn = np.sum((preds != c) & (targets == c))
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
            f1s.append(f1)
        return np.mean(f1s) if len(f1s) > 0 else 0.0

## 5. Training and Validation Epochs
Functions to execute a single epoch of training or validation.

In [22]:
def train_epoch(model, dataloader, criterion_group, optimizer, device, scheduler=None):
    model.train()
    epoch_group_loss = 0.0
    
    group_correct = 0
    group_total = 0
    
    all_group_preds = []
    all_group_labels = []
    
    loop = tqdm(dataloader, desc="Training", leave=False)
    for images, annotations in loop:
        images = images.to(device)
        
        batch_group_labels = []
        for ann in annotations:
            batch_group_labels.append(ann['groupLabel_idx'])
            
        group_labels = torch.tensor(batch_group_labels, dtype=torch.long, device=device)
        
        optimizer.zero_grad()
        group_outputs, _ = model(images, annotations)
        
        loss = criterion_group(group_outputs, group_labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
        
        _, pred_group = torch.max(group_outputs, 1)
        group_correct += (pred_group == group_labels).sum().item()
        group_total += group_labels.size(0)
        
        all_group_preds.extend(pred_group.cpu().numpy())
        all_group_labels.extend(group_labels.cpu().numpy())
        
        epoch_group_loss += loss.item() * images.size(0)
        loop.set_postfix(loss=loss.item(), group_acc=group_correct/max(group_total, 1))
        
    num_samples = len(dataloader.dataset)
    group_f1 = compute_macro_f1(np.array(all_group_preds), np.array(all_group_labels))
    return {
        'loss': epoch_group_loss / num_samples,
        'acc': (group_correct / group_total) if group_total > 0 else 0.0,
        'f1': group_f1
    }


@torch.no_grad()
def val_epoch(model, dataloader, criterion_group, device):
    model.eval()
    epoch_group_loss = 0.0
    
    group_correct = 0
    group_total = 0
    
    all_group_preds = []
    all_group_labels = []
    
    for images, annotations in dataloader:
        images = images.to(device)
        
        batch_group_labels = []
        for ann in annotations:
            batch_group_labels.append(ann['groupLabel_idx'])
            
        group_labels = torch.tensor(batch_group_labels, dtype=torch.long, device=device)
        group_outputs, _ = model(images, annotations)
        
        loss = criterion_group(group_outputs, group_labels)
        
        _, pred_group = torch.max(group_outputs, 1)
        group_correct += (pred_group == group_labels).sum().item()
        group_total += group_labels.size(0)
        
        all_group_preds.extend(pred_group.cpu().numpy())
        all_group_labels.extend(group_labels.cpu().numpy())
        
        epoch_group_loss += loss.item() * images.size(0)
        
    num_samples = len(dataloader.dataset)
    group_f1 = compute_macro_f1(np.array(all_group_preds), np.array(all_group_labels))
    return {
        'loss': epoch_group_loss / num_samples,
        'acc': (group_correct / group_total) if group_total > 0 else 0.0,
        'f1': group_f1
    }

## 6. Training Pipeline (Main Runner)
Define settings, prepare data augmentations, load validation checkpoints if available, and run the main training and validation loops.

In [23]:
def run_baseline_3():
    epochs = 5
    batch_size = 4
    lr = 1e-4
    data_path = '/kaggle/input/datasets/ahmedmohamed365/volleyball/volleyball_/videos'
    # dry_run is disabled as mock dataset logic was removed
    
    if not os.path.exists(data_path):
        data_path = '../volleyball/volleyball_/videos'
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"Dataset path '{data_path}' not found. Stopping program.")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running on device: {device}")

    train_transform = transforms.Compose([
        transforms.Resize((180, 320)),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    val_transform = transforms.Compose([
        transforms.Resize((180, 320)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    def collate_fn(batch):
        images = [item[0] for item in batch]
        annotations = [item[1] for item in batch]
        images = torch.stack(images, 0)
        return images, annotations
    
    train_dataset = VolleyBallDataset(split='train', transform=train_transform, data_path=data_path, seq_len=9, stride=3)
    val_dataset = VolleyBallDataset(split='val', transform=val_transform, data_path=data_path, seq_len=9, stride=3)
    test_dataset = VolleyBallDataset(split='test', transform=val_transform, data_path=data_path, seq_len=9, stride=3)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0, pin_memory=True)

    print(f"Train samples: {len(train_dataset)}")
    print(f"Validation samples: {len(val_dataset)}")
    print(f"Test samples: {len(test_dataset)}")

    model = GroupActivityRecognitionB3(
        num_group_classes=len(GROUP_ACTIVITIES),
        num_action_classes=len(PLAYER_ACTIONS),
        embed_dim=2048,
        hidden_size=512,
        num_layers=1,
        dropout=0.3
    )
    model = model.to(device)
    
    backbone_params = []
    head_params = []
    for name, param in model.named_parameters():
        if 'classifier' in name or 'lstm' in name:
            head_params.append(param)
        else:
            backbone_params.append(param)
            
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': lr, 'weight_decay': 1e-4},
        {'params': head_params, 'lr': lr * 10, 'weight_decay': 1e-3}
    ])
    
    criterion_group = nn.CrossEntropyLoss(label_smoothing=0.1)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    os.makedirs('checkpoints', exist_ok=True)
    best_model_path = 'checkpoints/best_model.pth'
    final_model_path = 'checkpoints/final_model.pth'
    best_val_f1 = 0.0

    if os.path.exists(best_model_path):
        print(f"Loading weights from existing best model checkpoint: {best_model_path}")
        try:
            model.load_state_dict(torch.load(best_model_path, map_location=device))
            print("Evaluating loaded model on validation split...")
            initial_val_metrics = val_epoch(model, val_loader, criterion_group, device)
            best_val_f1 = initial_val_metrics['f1']
            print(f"Loaded model validation F1: {best_val_f1:.4f}. Reset best_val_f1 to match.")
        except Exception as e:
            print(f"Warning: could not load or evaluate best checkpoint: {e}")
    elif os.path.exists(final_model_path):
        print(f"Loading weights from existing final model checkpoint: {final_model_path}")
        try:
            model.load_state_dict(torch.load(final_model_path, map_location=device))
            print("Evaluating loaded model on validation split...")
            initial_val_metrics = val_epoch(model, val_loader, criterion_group, device)
            best_val_f1 = initial_val_metrics['f1']
            print(f"Loaded model validation F1: {best_val_f1:.4f}. Reset best_val_f1 to match.")
        except Exception as e:
            print(f"Warning: could not load or evaluate final checkpoint: {e}")

    history = {
        'train_loss': [], 'train_acc': [], 'train_f1': [],
        'val_loss': [], 'val_acc': [], 'val_f1': []
    }
    
    print("\nStarting Training Loop...")
    for epoch in range(1, epochs + 1):
        print(f"\n--- Epoch {epoch}/{epochs} ---")
        current_lrs = [group['lr'] for group in optimizer.param_groups]
        print(f"Learning Rates - Backbone: {current_lrs[0]:.2e} | Head: {current_lrs[1]:.2e}")
        
        train_metrics = train_epoch(model, train_loader, criterion_group, optimizer, device)
        val_metrics = val_epoch(model, val_loader, criterion_group, device)
        
        scheduler.step()
        
        history['train_loss'].append(train_metrics['loss'])
        history['train_acc'].append(train_metrics['acc'])
        history['train_f1'].append(train_metrics['f1'])
        
        history['val_loss'].append(val_metrics['loss'])
        history['val_acc'].append(val_metrics['acc'])
        history['val_f1'].append(val_metrics['f1'])
        
        print(f"Train Loss: {train_metrics['loss']:.4f} | Train Acc: {train_metrics['acc']*100:.2f}% | Train F1: {train_metrics['f1']:.4f}")
        print(f"Val   Loss: {val_metrics['loss']:.4f} | Val   Acc: {val_metrics['acc']*100:.2f}% | Val   F1: {val_metrics['f1']:.4f}")

        if val_metrics['f1'] > best_val_f1:
            best_val_f1 = val_metrics['f1']
            torch.save(model.state_dict(), best_model_path)
            print(f"Saved new best model to {best_model_path} with Val F1: {best_val_f1:.4f}")

    torch.save(model.state_dict(), final_model_path)
    print(f"\nFinal model saved to {final_model_path}")
    print("Training completed successfully!")

## 7. Run Training Pipeline
Call the runner function to start the training process.

In [24]:
run_baseline_3()

Running on device: cuda
Train samples: 2152
Validation samples: 1341
Test samples: 1337

Starting Training Loop...

--- Epoch 1/5 ---
Learning Rates - Backbone: 1.00e-04 | Head: 1.00e-03


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 1.3007 | Train Acc: 63.38% | Train F1: 0.2461
Val   Loss: 1.0702 | Val   Acc: 67.34% | Val   F1: 0.4160
Saved new best model to checkpoints/best_model.pth with Val F1: 0.4160

--- Epoch 2/5 ---
Learning Rates - Backbone: 9.05e-05 | Head: 9.05e-04


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 1.0062 | Train Acc: 73.84% | Train F1: 0.6118
Val   Loss: 0.9723 | Val   Acc: 78.90% | Val   F1: 0.7353
Saved new best model to checkpoints/best_model.pth with Val F1: 0.7353

--- Epoch 3/5 ---
Learning Rates - Backbone: 6.58e-05 | Head: 6.55e-04


...

Train Loss: 0.8294 | Train Acc: 86.11% | Train F1: 0.8308
Val   Loss: 1.0125 | Val   Acc: 79.57% | Val   F1: 0.7578
Saved new best model to checkpoints/best_model.pth with Val F1: 0.7578

--- Epoch 4/5 ---
Learning Rates - Backbone: 3.52e-05 | Head: 3.46e-04


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 0.7025 | Train Acc: 91.64% | Train F1: 0.9026
Val   Loss: 0.9748 | Val   Acc: 82.70% | Val   F1: 0.8016
Saved new best model to checkpoints/best_model.pth with Val F1: 0.8016

--- Epoch 5/5 ---
Learning Rates - Backbone: 1.05e-05 | Head: 9.64e-05


Training:   0%|          | 0/538 [00:00<?, ?it/s]

Train Loss: 0.6049 | Train Acc: 95.03% | Train F1: 0.9443
Val   Loss: 0.9755 | Val   Acc: 83.37% | Val   F1: 0.8005

Final model saved to checkpoints/final_model.pth
Training completed successfully!


In [25]:
!zip -r /kaggle/working/checkpoints.zip /kaggle/working/checkpoints

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/final_model.pth (deflated 7%)
  adding: kaggle/working/checkpoints/best_model.pth (deflated 7%)
